<a href="https://colab.research.google.com/github/officiallong/BUS4-118S-build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Laptop_Refund_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laptop Refund Agent — Google Gemini + LangGraph

## 1. Install libraries

In [1]:
%pip install -q -U 'langchain>=1,<2' 'langchain-google-genai>=4,<5' 'langgraph>=1,<2'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 2. Connect to Gemini

In [2]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from google import genai

# Get Gemini API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Connect to Google Gemini
client = genai.Client(api_key=api_key)

# Use the model recommended by Google's API
model_name = "gemini-3.8-flash"

# Set up the Gemini model
model = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=api_key,
    temperature=0
)

# Test Gemini connection
print("Trying Gemini model:", model_name)

response = model.invoke("Hello!")
print("Gemini response:", response.content)

print("Gemini connection successful!")

Trying Gemini model: gemini-3.8-flash
Gemini response: [{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'Eo4ECosEAWkUfRNX131/nhX35FZQ/aJCPe2HX+pSoQWJvTMIJcv53+xlGkiZMynX/sFHE418sGB775PRQ14zixpaYsUEfl0atiMnZeuI0IEcYdTLrpszhrB8EHz1xX7/+DVYear9PtDjoPr82cYexKAbERNdrHwMcxWfEfkQivHMYOeJMNipR9dh5hLaFukRlrt/htOZv7aA2vF/1rQT7ALJCp8SnDko5tDDiWHdkuVnPKDcxONeErLpF988rEXEYTpz+/Tok3LovPZsBa2Uulq9dpBKbfRjIgmKZ4R+CC95eqZ26mnnnSjBPlKsCBFK0DrrOMUeQDYvo6hiQEOtWopuUvPkNXLl1+SesZb0++rh6e21w13SIGrSNWWEYrR1lnzdpkxvkWyopqS6xxL8v7oFle1gtc5kI/5b83lzfUgz9G8OvTqa1S7VmsCZ5vyWHTXMU+T2Wb9pFjw8mRwyvbTvIqF2dDq3ADkDp2Ourx4NQB+KupGkMN1R96YRb0VjkHiBY5l6L6RgXNOxVgKG2PPsLak78YMWv6/z+sYkoJ9kdKuM/oUrWbU0UsfjgnH2YkQf3wx49dgmbT+J7lId3HVHChlht3FwzQYlztk1tA/2b2lSfoV0hfepvXqzldz+x+sNZV8W5IckMO/zox0HLKiN0ObYR3O+waXhF68vLaTLNWmh5AWrnohuhtbb6j1SRA=='}}]
Gemini connection successful!


## 3. Refund tools

In [3]:
from langchain_core.tools import tool

SAMPLE_RETURN_DAYS = 30

@tool
def check_refund_policy() -> str:
    """Look up the sample laptop store refund policy, return window, and refund procedure."""
    return (
        "SAMPLE policy: Returns within 30 days of delivery; laptop must be in good "
        "condition; proof of purchase required; refunds use the original payment "
        "method. Contact customer support to request a return."
    )

@tool
def check_refund_eligibility(days_since_delivery: int, good_condition: bool, has_proof_of_purchase: bool) -> str:
    """Check sample refund eligibility using days since delivery, laptop condition, and receipt. Never guess missing information."""
    if days_since_delivery < 0:
        return "Invalid days_since_delivery; ask the customer to clarify."
    problems = []
    if days_since_delivery > SAMPLE_RETURN_DAYS:
        problems.append("more than 30 days since delivery")
    if not good_condition:
        problems.append("laptop not in good condition")
    if not has_proof_of_purchase:
        problems.append("proof of purchase missing")
    if problems:
        return "Not eligible under SAMPLE policy: " + "; ".join(problems)
    return "Meets SAMPLE eligibility criteria. Contact support to request a return; no refund has been approved or processed."

print("Refund tools ready.")


Refund tools ready.


## 4. Refund Agent

In [4]:
from langchain.agents import create_agent

agent_tools = [check_refund_policy, check_refund_eligibility]
system_prompt = """You are a laptop store Refund Agent for a classroom demo.
All refund rules are SAMPLE policies, not a real store's official policy.
Use your refund tools to answer policy questions and check hypothetical eligibility.
Never invent policy terms or claim a refund was approved, issued, or processed.
Ask for missing eligibility details instead of guessing them.
Be clear, polite, and concise. Only handle refunds and returns.
"""
agent_graph = create_agent(model=model, tools=agent_tools, system_prompt=system_prompt)
print("Refund Agent created successfully!")


Refund Agent created successfully!


## 5. Example 1 — Eligible return

In [5]:
inputs = {"messages": [("user", "Can I return my laptop 15 days after delivery if it's in good condition and I have my receipt?")]}
result = agent_graph.invoke(inputs)
print("Agent returned:", result["messages"][-1].content)
print("\nStep by Step execution:")
for message in result["messages"]:
    print(message.pretty_repr())


Agent returned: [{'type': 'text', 'text': 'Based on our sample policy, your return meets the sample eligibility criteria since it is within the return window, in good condition, and you have your receipt. \n\nPlease note that this does not mean a refund has been approved, issued, or processed. You would need to contact support to formally initiate a return request.', 'extras': {'signature': 'EvYDCvMDAWkUfRNuENelbM9wLlZSTg4M+5SV4gnWqp4sbPmygRgjzqvNsi3Ge/HcsZUOvwE0QD45JmJ0OrCAFdhzOqIfEf3WCHfrsHQmfAZWk3UhvoWaYBYPNKHpBolNM1/Z6XhGpUkKJ1pxB+M1wn3ooZQA9vE11NR9vOFgYaGp/ThASERYfWSVQQvhDvnN1eSIQJuoXwMo1LsHsEVWjIu5l7oMXYwK8Le6GrQO9MONq9idPgLoduNdAh8VtTirANAN3KOIflr7zcuTOyzYFXU4w3dXW568FkkXRfi9CconTxXF36odFzt75kvkYGsYQ/0ZZRuu7IVx3sKAoxAov8ZhyT7eh2XCKIbccPfMPOM+GA64sOyrMkjWpEcB1EoHsW0JGbPsEOUqU0va2W4dvCPquTYVBJqMYzubZBfMN/NO8GMp3GmtL8d56UXuvLhbCq9Lrozz5/2DfX0IPKMAuUE5g9qybv49/DmXSngL3Z4krn6Nu+0QFzU4cA8KYMym+5E04nAeIoMcsBHpxGJnZg6A5Q+KTdRh2WmKtqTsSALlusBIdSamqpSsfpiSjUkHGZPRybaXhF2TcFriWq5qRaT44pI1g

## 6. Example 2 — Past the return deadline

In [6]:
result2 = agent_graph.invoke({"messages": [("user", "My laptop arrived 45 days ago. It is in good condition and I have the receipt. Can I return it?")]})
print("Agent returned:", result2["messages"][-1].content)


Agent returned: [{'type': 'text', 'text': 'Based on our sample refund policy, your laptop is not eligible for a return because it has been 45 days since delivery, which exceeds the standard 30-day return window. \n\nPlease note that these are sample rules for demonstration purposes. Let me know if you have any other questions regarding sample refund policies!', 'extras': {'signature': 'EqUDCqIDAWkUfRNDXV3MfBG74wqRs/PPOYYQRpXoc3K78NPlP+LtFwnqYpwfcJ5FwFPq428H2kMR06k0yHwOZpZlFEGTb37VIQS6fva5P3oXcL/gtuCWApJzVYY7iE51uJ/Ou8x1FZqQYofSqABGRvC/UgS8uuvUxQ16TiJwtb0xrRx3m+8/WnZOK+w6iaJgT3cv5GK9OpCYiS2wyvLj9smejCIWSR0fhKiUbqDq130PkNGifi5JjjcCL921lNzuc9vFNeF//pFUwBMtydHX0/HH5MsIi4/FE4uDLJRAAeSDKTrgC4U9ZPFMA/LzkYyeGSfhxeMQYgQ2IdkD8CG0u7tQkf19Mhygs0v8cPm3QMaIcteS1aJUnrlSltzqB0mNhqexWc/VBW0wrlWmzJOe03Ah4JCqlErvtsilvUeQCIQIGTu9gAF1GKjgevA7QBp5Z16d100Z6qmAnOVjrMBiDIS2nkTrYs7E39BeDnPas5+zxMx55hNL2XmqLi/kB5NVclUL39Y0XJre+OJQuSPIJ69V28TA3X+KEkD2JUsbjyIdyF4GMFqmpQ=='}}]


## 7. Example 3 — Missing details

In [7]:
result3 = agent_graph.invoke({"messages": [("user", "Can I get a refund for my laptop?")]})
print("Agent returned:", result3["messages"][-1].content)


Agent returned: [{'type': 'text', 'text': 'Under our sample policy, laptops may be eligible for a refund if they meet the following criteria:\n* The return is requested within 30 days of delivery.\n* The laptop is in good condition.\n* You have a valid proof of purchase (such as a receipt or order confirmation).\n\nTo check your eligibility, could you please let me know:\n1. How many days have passed since the laptop was delivered?\n2. Is the laptop in good condition?\n3. Do you have proof of purchase?', 'extras': {'signature': 'EpQECpEEAWkUfROaxT0Ce6iRaAeR+b0r2P/uGk5lfz3YNckazpyL0sXC2PkrsFW9XK2gItif3EUgoJDE01UVvY+pz9SQmzUJh6H5MCIde6DNHcjh8GLDP6EAfUdG5nNvidHBJ1SsKQZ+Hv6e2KeX+LQKDtKq7kgUbAWMK0rYTfmLJE61CQT5lT3tAcZ1FHCHmRpCXpgIGQhLckrqjZ5xcqabL8iDqaOekKF4P7ezJ4oaKbef8KJzd5kkrRw4Z24i6mhySy4h7A9Q8vcn2pJsUE20Go33Ogufbhtn7pPW52H4d/Q4LnLT9ukVqHsGebN5DaK7TZ4sokf9UZDBP3Dal+8X9uUoWfFRVJkIoC2lUdDWYzby15haf4IcGEuxG2ZZ1vjhA0cmrMZ3t3IsnrPo2NdWx5eny0bkvylSXsKjdQg6PvMlHztlziKsFmkKmujmtqsVsT67h0QbdYAsF

## 8. Three-turn conversation memory

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

refund_agent_with_memory = create_agent(
    model=model, tools=agent_tools, system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "refund-demo"}}
for question in [
    "My laptop arrived 12 days ago.",
    "It's in good condition, and I have my receipt.",
    "Can I return it under the sample policy?",
]:
    conversation = refund_agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]}, config=config
    )
    print("Customer:", question)
    print("Agent:", conversation["messages"][-1].content, "\n")


Customer: My laptop arrived 12 days ago.
Agent: [{'type': 'text', 'text': 'To help determine your refund eligibility, could you please confirm:\n\n1. Is the laptop in good condition (undamaged, with original packaging and accessories)?\n2. Do you have proof of purchase (such as a receipt or order confirmation)?', 'extras': {'signature': 'EqAICp0IAWkUfROEBXgjWgThDyX0H/9OTPex1JQNf9eQsw1op9kS4GnVs7ZbfEd7w8e/dT1czp3HiXGwCf1JuD7BDlwKwC52JB9uDksaNs3S9n3JnVxr4yYLbrEpz0wskiDwLk/7Hjarfb9cohgkMdY+lMtQ7xiTHEjR2sl8PzZP580AOo2S8WExbO64mRonZqb7CLYakFauCRnb6vI8y6wK1wmBL8aX3DqKsZVywIjNG2aT77zawPo83iruuMqotZZJ3UEEpJ3GQAIalBwtC3COehX+2Jb0Y7PFPZLRFJojAU4Nq2xyteYYdrp9woJ72vAC6QzSHD+OCNUg1fEmROejq67TkaEKTgtOSk17NrSiCYEqkCGIyYE1WvHKwAlifN8o6jd/jXSi7CS1mWqYFORozBktf5wty1hemvvYwifbpUASnkvrnM/+35arTk8OLQd+Pa/XzZuTQCc+YtzniueeFAGRY2dugsNrYhM/qUKU+w8D9iRYHD5zPO1nECiP5UFOAjIoVQMiUfDjr+gyP2YmjJRzZI3WHR7+qpJ2adUFMRRv3QmPVd0t68RiBuUphRztur0N8WJ04DV+qxITJZANMIM4Sv84ylrpK9QhPtVaJN4OaR098gh80pUqeSCKF8dm4A5SzTt1YIFDXUHR